# B2.10 · Choosing the model backbone

**Function B — Application Security with an AI SDLC → The Harness that Runs the SDLC**  ·  *AI for Security*

Builds on **[B2.9 · Building a domain harness: one skeleton, four oracles](https://spbreed.github.io/cyber-commons/lessons/B2.9.html)**.

| | |
|---|---|
| Open-source tooling | LiteLLM, vLLM, Ollama |
| Open-weight models | Kimi K2.7-Code, GLM-5.2, Llama 4 |
| Frontier models | Claude Opus 5 |

> **Runs anywhere.** Every line of code is in this notebook — nothing to install, nothing to clone, no API key, no network. Standard library only, so it works on a Kaggle kernel with the internet switched off — and where a lesson involves a model, the same code calls an open-weight endpoint or a frontier API when you configure one.

## 1 · The hook

The vendor chart was not run on your corpus, at your context length, with your tools. Choosing the backbone on someone else's benchmark is choosing on evidence about a different system.

## 2 · The framework

```
   the vendor chart                  your decision
   +--------------------+            +---------------------------+
   | someone's corpus   |            | your corpus               |
   | their context len  |    vs      | your context length       |
   | their tools        |            | your tools, your budget   |
   +--------------------+            +---------------------------+

   frontier | open-weight hosted | self-hosted open-weight
   the axis that usually decides is not capability, it is where data may go
```

Two questions get confused here, and only one of them is worth your time.

The first is *which model is best for security work right now*. As of writing:
Kimi K2.6 and K2.7-Code hold up well on long-horizon repository work, the GLM-5.x
line performs strongly on vulnerability discovery, and frontier models still lead
on deep exploitation chains. That paragraph will be wrong within a quarter, and
anything built on it will be wrong with it.

The second question is the one that lasts: **can you substitute the backbone
without rewriting the harness?** If swapping a model means touching prompt
assembly, tool schemas, parsing and retry logic, then the honest answer is that
you did not choose a model — you married one.

So the discipline is:

- Evaluate on **your corpus**, not a vendor chart. The chart measures a
  distribution that is not yours, on tasks that are not yours.
- Put the backbone behind an **interface** — one adapter, one place to change.
- Keep the eval so a substitution is a measurement, not an argument.
- Weigh what the chart never shows: data sovereignty, cost per finding, and
  whether your estate is allowed to send code to that endpoint at all.

## 3 · An interface, so the backbone is a parameter

Three stand-in backbones with deliberately different behaviour. None is a language model; each stands in for one so substitution is visible.

In [ ]:
CORPUS = [
 # (unit, true_cwe)
 ("get_report",   "CWE-22"), ("run_export", "CWE-78"), ("safe_query", None),
 ("legacy_dump",  "CWE-89"), ("render_row", None),     ("admin_purge", "CWE-78"),
]

def backbone_a(unit):                 # strong on injection, misses traversal
    return {"run_export": "CWE-78", "legacy_dump": "CWE-89",
            "admin_purge": "CWE-78"}.get(unit)
def backbone_b(unit):                 # broad recall, some false positives
    return {"get_report": "CWE-22", "run_export": "CWE-78", "legacy_dump": "CWE-89",
            "admin_purge": "CWE-78", "render_row": "CWE-79"}.get(unit)
def backbone_c(unit):                 # conservative
    return {"run_export": "CWE-78"}.get(unit)

BACKBONES = {"kimi-k2.6-stand-in": backbone_a,
             "glm-5.2-stand-in":   backbone_b,
             "small-local-stand-in": backbone_c}

def harness(find, corpus):
    """The harness. Note it takes `find` as an argument - that is the point."""
    return [(u, find(u)) for u, _ in corpus if find(u)]

for name in sorted(BACKBONES):
    out = harness(BACKBONES[name], CORPUS)
    print(f"{name:22s}{len(out)} findings")

## 4 · Score them on your corpus, not on a chart

In [ ]:
def score(find, corpus, cost_per_call=0.004):
    truth = dict(corpus)
    found = harness(find, corpus)
    tp = [u for u, c in found if truth.get(u) == c]
    fp = [u for u, c in found if truth.get(u) != c]
    planted = [u for u, c in corpus if c]
    fn = [u for u in planted if u not in [x for x, _ in found]]
    recall = len(tp) / len(planted)
    prec = len(tp) / len(found) if found else 0.0
    spend = len(corpus) * cost_per_call
    return {"recall": recall, "precision": prec, "tp": len(tp), "fp": len(fp),
            "fn": len(fn), "spend": spend,
            "cost_per_tp": (spend / len(tp)) if tp else float("inf")}

print(f"{'backbone':22s}{'recall':>8}{'prec':>7}{'tp':>4}{'fp':>4}{'fn':>4}{'$/finding':>11}")
results = {}
for name in sorted(BACKBONES):
    s = score(BACKBONES[name], CORPUS)
    results[name] = s
    print(f"{name:22s}{s['recall']:>7.0%}{s['precision']:>7.0%}"
          f"{s['tp']:>4}{s['fp']:>4}{s['fn']:>4}{s['cost_per_tp']:>10.3f}")
best_recall = max(results, key=lambda n: (results[n]["recall"], n))
best_cost = min(results, key=lambda n: (results[n]["cost_per_tp"], n))
print(f"\nbest recall        : {best_recall}")
print(f"best cost/finding  : {best_cost}")
print("They are not the same backbone, and which one you want depends on")
print("whether an analyst reviews the output or a ticket is opened from it.")

## 5 · Where it breaks — the harness that married its model

In [ ]:
def coupled_harness(unit, backbone_name):
    """Prompt assembly, parsing and retry all keyed to one vendor's quirks."""
    if backbone_name == "kimi-k2.6-stand-in":
        raw = backbone_a(unit)
        return raw                                   # returns a bare CWE
    if backbone_name == "glm-5.2-stand-in":
        raw = backbone_b(unit)
        return {"cwe": raw} if raw else None         # returns an object
    raise KeyError(f"no parsing branch for {backbone_name}")

for name in sorted(BACKBONES):
    try:
        out = [u for u, _ in CORPUS if coupled_harness(u, name)]
        print(f"   {name:22s}{len(out)} findings")
    except KeyError as e:
        print(f"   {name:22s}FAILS: {e}")
print()
print("Adding a third backbone to the coupled harness is a code change in the")
print("parser, the prompt and the retry path. Adding it to the harness above is")
print("a dictionary entry. Same models, same corpus - the difference is where")
print("the vendor's shape was allowed to leak to.")

## 6 · The control — substitute behind an unchanged interface

In [ ]:
def substitute(harness_fn, corpus, frm, to):
    before = score(BACKBONES[frm], corpus)
    after  = score(BACKBONES[to], corpus)
    return {"from": frm, "to": to,
            "recall": (before["recall"], after["recall"]),
            "precision": (before["precision"], after["precision"]),
            "cost_per_tp": (round(before["cost_per_tp"], 3), round(after["cost_per_tp"], 3)),
            "harness_changed": False}

sw = substitute(harness, CORPUS, "kimi-k2.6-stand-in", "glm-5.2-stand-in")
for k, v in sw.items():
    print(f"   {k:16s}{v}")
print()
print("Recall up, precision down, cost per finding down. That is a decision with")
print("numbers behind it, taken in one line, and reversible in one line.")
assert sw["harness_changed"] is False

## 7 · Verify — the constraints the chart never shows

In [ ]:
CONSTRAINTS = {
 "kimi-k2.6-stand-in":   {"weights": "open", "self_hostable": True,  "code_leaves_estate": False},
 "glm-5.2-stand-in":     {"weights": "open", "self_hostable": True,  "code_leaves_estate": False},
 "frontier-api-stand-in":{"weights": "closed","self_hostable": False, "code_leaves_estate": True},
}
print(f"{'backbone':24s}{'weights':9s}{'self-host':11s}source code leaves the estate")
for n in sorted(CONSTRAINTS):
    c = CONSTRAINTS[n]
    print(f"{n:24s}{c['weights']:9s}{str(c['self_hostable']):11s}{c['code_leaves_estate']}")
print()
eligible = [n for n, c in sorted(CONSTRAINTS.items()) if not c["code_leaves_estate"]]
print(f"eligible where source may not leave the estate: {eligible}")
print()
print("For a great many organisations this single column removes the top of")
print("every published leaderboard before accuracy is discussed at all - which")
print("is the strongest argument for designing the substitution seam early.")
assert "frontier-api-stand-in" not in eligible

## What you just proved

Three stand-in backbones are scored on the same corpus: the one with the best recall is not the one with the best cost per finding. A harness that couples to vendor output shapes fails outright on the third backbone, while the interface version substitutes in a single line and reports the recall, precision and cost deltas. A data-sovereignty column then removes the closed-weights option entirely.

## Your turn

Time how long it takes to swap the backbone in your own harness. If it is more than an afternoon, that number — not a benchmark — is what will decide your model choice for the next two years.

---

**Next → [B2.11 · Evaluating a security harness](https://spbreed.github.io/cyber-commons/lessons/B2.11.html)**

[All lessons](https://spbreed.github.io/cyber-commons/lessons/) · [This lesson's page](https://spbreed.github.io/cyber-commons/lessons/B2.10.html) · [Source](https://github.com/spbreed/cyber-commons/blob/claude/vulnbench-setup-scheduling-81aqov/labs/notebooks/B2.10.ipynb)

*Cyber Commons — a free, open commons for Cyber AI.*